# **Phase 0 – Project Setup**

**Import Required Libraries**

In [ ]:
import os
import glob
import numpy as np
import pandas as pd

from pathlib import Path

import warnings
warnings.filterwarnings("ignore")

**Github Repo Connection**

In [ ]:
!git clone https://github.com/edusatyaki/ProjectDrive.git

Cloning into 'ProjectDrive'...
remote: Enumerating objects: 252, done.
remote: Counting objects: 100% (252/252), done.
remote: Compressing objects: 100% (133/133), done.
remote: Total 252 (delta 101), reused 171 (delta 66), pack-reused 0 (from 0)
Receiving objects: 100% (252/252), 42.21 MiB | 8.98 MiB/s, done.
Resolving deltas: 100% (101/101), done.
Filtering content: 100% (12/12), 791.52 MiB | 55.87 MiB/s, done.


**Raw Dataset Path Connection**

In [ ]:
from pathlib import Path
import glob

RAW_FOLDER = Path("ProjectDrive/Group-C/NYC Yellow Taxi 2025/Data Cleaning/Raw Dataset")

files = sorted(glob.glob(str(RAW_FOLDER / "*.parquet")))

print(f"Total files: {len(files)}")

for file in files:
    print(Path(file).name)

Total files: 12
yellow_tripdata_2025-01.parquet
yellow_tripdata_2025-02.parquet
yellow_tripdata_2025-03.parquet
yellow_tripdata_2025-04.parquet
yellow_tripdata_2025-05.parquet
yellow_tripdata_2025-06.parquet
yellow_tripdata_2025-07.parquet
yellow_tripdata_2025-08.parquet
yellow_tripdata_2025-09.parquet
yellow_tripdata_2025-10.parquet
yellow_tripdata_2025-11.parquet
yellow_tripdata_2025-12.parquet


In [ ]:
!ls

ProjectDrive  sample_data


**Cleaned Dataset Path**

In [ ]:
CLEANED_FOLDER = Path("ProjectDrive/Group-C/NYC Yellow Taxi 2025/Data Cleaning/Cleaned Dataset")

**Inspect One Month Dataset**

In [ ]:
import pandas as pd

df = pd.read_parquet(files[0])

print(df.shape)

df.head()

(3475226, 20)


,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,cbd_congestion_fee
0,1,2025-01-01 00:18:38,2025-01-01 00:26:59,1.0,1.60,1.0,N,229,237,1,10.0,3.5,0.5,3.00,0.0,1.0,18.00,2.5,0.0,0.0
1,1,2025-01-01 00:32:40,2025-01-01 00:35:13,1.0,0.50,1.0,N,236,237,1,5.1,3.5,0.5,2.02,0.0,1.0,12.12,2.5,0.0,0.0
2,1,2025-01-01 00:44:04,2025-01-01 00:46:01,1.0,0.60,1.0,N,141,141,1,5.1,3.5,0.5,2.00,0.0,1.0,12.10,2.5,0.0,0.0
3,2,2025-01-01 00:14:27,2025-01-01 00:20:01,3.0,0.52,1.0,N,244,244,2,7.2,1.0,0.5,0.00,0.0,1.0,9.70,0.0,0.0,0.0
4,2,2025-01-01 00:21:34,2025-01-01 00:25:06,3.0,0.66,1.0,N,244,116,2,5.8,1.0,0.5,0.00,0.0,1.0,8.30,0.0,0.0,0.0


**Dataset Information**

In [ ]:
df.info()

df.describe()

df.columns

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3475226 entries, 0 to 3475225
Data columns (total 20 columns):
 #   Column                 Dtype         
---  ------                 -----         
 0   VendorID               int32         
 1   tpep_pickup_datetime   datetime64[us]
 2   tpep_dropoff_datetime  datetime64[us]
 3   passenger_count        float64       
 4   trip_distance          float64       
 5   RatecodeID             float64       
 6   store_and_fwd_flag     object        
 7   PULocationID           int32         
 8   DOLocationID           int32         
 9   payment_type           int64         
 10  fare_amount            float64       
 11  extra                  float64       
 12  mta_tax                float64       
 13  tip_amount             float64       
 14  tolls_amount           float64       
 15  improvement_surcharge  float64       
 16  total_amount           float64       
 17  congestion_surcharge   float64       
 18  Airport_fee           

Index(['VendorID', 'tpep_pickup_datetime', 'tpep_dropoff_datetime',
       'passenger_count', 'trip_distance', 'RatecodeID', 'store_and_fwd_flag',
       'PULocationID', 'DOLocationID', 'payment_type', 'fare_amount', 'extra',
       'mta_tax', 'tip_amount', 'tolls_amount', 'improvement_surcharge',
       'total_amount', 'congestion_surcharge', 'Airport_fee',
       'cbd_congestion_fee'],
      dtype='object')

**Missing Values**

In [ ]:
df.isnull().sum()

,0
VendorID,0
tpep_pickup_datetime,0
tpep_dropoff_datetime,0
passenger_count,540149
trip_distance,0
RatecodeID,540149
store_and_fwd_flag,540149
PULocationID,0
DOLocationID,0
payment_type,0


**Duplicate Data**

In [ ]:
df.duplicated().sum()

np.int64(0)

# **Phase 1 : Data Cleaning**

**Clean Every Month Dataset**

In [ ]:
files = sorted(glob.glob(str(RAW_FOLDER / "*.parquet")))

for file in files:

    print(f"Processing {os.path.basename(file)}")

    df = pd.read_parquet(file)

    # Convert datetime
    df["tpep_pickup_datetime"] = pd.to_datetime(df["tpep_pickup_datetime"])
    df["tpep_dropoff_datetime"] = pd.to_datetime(df["tpep_dropoff_datetime"])

    # Trip Duration
    df["trip_duration"] = (
        df["tpep_dropoff_datetime"] -
        df["tpep_pickup_datetime"]
    ).dt.total_seconds() / 60

    # Remove invalid rows
    df = df[df["fare_amount"] > 0]
    df = df[df["trip_distance"] > 0]
    df = df[df["trip_duration"] > 0]

    # Remove duplicates
    df = df.drop_duplicates()

    # Remove outliers
    df = df[df["trip_duration"] <= 300]
    df = df[df["trip_distance"] <= 100]
    df = df[df["fare_amount"] <= 500]

    # Passenger count
    df = df[
        (df["passenger_count"] >= 1) &
        (df["passenger_count"] <= 6)
    ]

    # =====================
    # Feature Engineering
    # =====================

    df["pickup_hour"] = df["tpep_pickup_datetime"].dt.hour

    df["pickup_day"] = df["tpep_pickup_datetime"].dt.day_name()

    df["pickup_month"] = df["tpep_pickup_datetime"].dt.month_name()

    df["pickup_year"] = df["tpep_pickup_datetime"].dt.year

    df["is_weekend"] = (
        df["tpep_pickup_datetime"].dt.weekday >= 5
    )

    df["fare_per_mile"] = (
        df["fare_amount"] /
        df["trip_distance"]
    )

    df["tip_pct"] = (
    df["tip_amount"] /
    df["fare_amount"] * 100
    )

    df["tip_pct"] = (
        df["tip_pct"]
          .replace([np.inf, -np.inf], 0)
          .fillna(0)
    )

    # Save
    filename = Path(file).stem + "_cleaned.parquet"

    output_path = CLEANED_FOLDER / filename

    df.to_parquet(output_path, index=False)

    print(f"Saved : {filename}")

    del df

Processing yellow_tripdata_2025-01.parquet
Saved : yellow_tripdata_2025-01_cleaned.parquet
Processing yellow_tripdata_2025-02.parquet
Saved : yellow_tripdata_2025-02_cleaned.parquet
Processing yellow_tripdata_2025-03.parquet
Saved : yellow_tripdata_2025-03_cleaned.parquet
Processing yellow_tripdata_2025-04.parquet
Saved : yellow_tripdata_2025-04_cleaned.parquet
Processing yellow_tripdata_2025-05.parquet
Saved : yellow_tripdata_2025-05_cleaned.parquet
Processing yellow_tripdata_2025-06.parquet
Saved : yellow_tripdata_2025-06_cleaned.parquet
Processing yellow_tripdata_2025-07.parquet
Saved : yellow_tripdata_2025-07_cleaned.parquet
Processing yellow_tripdata_2025-08.parquet
Saved : yellow_tripdata_2025-08_cleaned.parquet
Processing yellow_tripdata_2025-09.parquet
Saved : yellow_tripdata_2025-09_cleaned.parquet
Processing yellow_tripdata_2025-10.parquet
Saved : yellow_tripdata_2025-10_cleaned.parquet
Processing yellow_tripdata_2025-11.parquet
Saved : yellow_tripdata_2025-11_cleaned.parquet

**Verify Cleaned Files**

In [ ]:
import glob

cleaned_files = sorted(glob.glob(str(CLEANED_FOLDER / "*.parquet")))

print(f"Total cleaned files: {len(cleaned_files)}")

Total cleaned files: 12


**Preview Cleaned Dataset**

In [ ]:
from pathlib import Path
import pandas as pd

CLEANED_FOLDER = Path(
    "ProjectDrive/Group-C/NYC Yellow Taxi 2025/Data Cleaning/Cleaned Dataset"
)

df = pd.read_parquet(
    CLEANED_FOLDER / "yellow_tripdata_2025-01_cleaned.parquet"
)

print(df.shape)

df.isnull().sum()

(2814358, 28)


,0
VendorID,0
tpep_pickup_datetime,0
tpep_dropoff_datetime,0
passenger_count,0
trip_distance,0
RatecodeID,0
store_and_fwd_flag,0
PULocationID,0
DOLocationID,0
payment_type,0


In [ ]:
from pathlib import Path
import pandas as pd

CLEANED_FOLDER = Path(
    "ProjectDrive/Group-C/NYC Yellow Taxi 2025/Data Cleaning/Cleaned Dataset"
)

df = pd.read_parquet(
    CLEANED_FOLDER / "yellow_tripdata_2025-02_cleaned.parquet"
)

print(df.shape)

df.isnull().sum()

(2655671, 28)


,0
VendorID,0
tpep_pickup_datetime,0
tpep_dropoff_datetime,0
passenger_count,0
trip_distance,0
RatecodeID,0
store_and_fwd_flag,0
PULocationID,0
DOLocationID,0
payment_type,0


# **Download Cleaned Dataset**

**Check where the cleaned files are**

In [ ]:
print(CLEANED_FOLDER.resolve())

/content/ProjectDrive/Group-C/NYC Yellow Taxi 2025/Data Cleaning/Cleaned Dataset


In [ ]:
import os

print(os.listdir(CLEANED_FOLDER))

['yellow_tripdata_2025-02_cleaned.parquet', 'yellow_tripdata_2025-01_cleaned.parquet', 'yellow_tripdata_2025-08_cleaned.parquet', 'yellow_tripdata_2025-10_cleaned.parquet', 'yellow_tripdata_2025-05_cleaned.parquet', '.gitkeep', 'yellow_tripdata_2025-11_cleaned.parquet', 'yellow_tripdata_2025-07_cleaned.parquet', 'yellow_tripdata_2025-03_cleaned.parquet', 'yellow_tripdata_2025-04_cleaned.parquet', 'yellow_tripdata_2025-09_cleaned.parquet', 'yellow_tripdata_2025-06_cleaned.parquet', 'yellow_tripdata_2025-12_cleaned.parquet']


**Create the ZIP**

In [ ]:
import shutil

shutil.make_archive(
    "Cleaned_Dataset",   # ZIP file name
    "zip",
    CLEANED_FOLDER
)

print("ZIP file created.")

ZIP file created.


**Verify the ZIP exists**

In [ ]:
import os

print(os.path.exists("Cleaned_Dataset.zip"))

True


# **End of Data Cleaning Notebook**